# Visualize

Visualize the effect of the rng_seed on the uncertainty quantification.

For a specified hurricane (Name), graphically compare the uncertainty ellipses across the different rng_seed values. An individual sub-plot is created for each {time, ftime} pair.

In [ ]:
%matplotlib inline
%load_ext autotime
import glob
import numpy as np
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

In [ ]:
FIGURE_PATH = "figures/analysis/"
PREDICTIONS_PATH = "saved_predictions/"

In [ ]:
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 300
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

In [ ]:
ELLIPSE_COLOR = {1: 'darkviolet', 2: 'goldenrod', 3: 'dodgerblue'}
NAUTICAL_MILE_TO_KM = 1.852
FIGURE_WIDTH = 32

In [ ]:
THETA = np.linspace(0, 2 * np.pi, 1000)

def plot_circle(ax, radius, color):
    x = radius * np.cos(THETA)
    y = radius * np.sin(THETA)
    ax.plot(x, y, '-', color=color, linewidth=4)


def plot_ellipse(ax, sigma_u, sigma_v, rho, color):
    # r = np.sqrt(-2.0 * (np.log(1.0 - 0.67))) for Pr(capture) = 67%
    r = 1.489068584398725

    x = r * sigma_u * np.cos(THETA)
    y = r * sigma_v * (rho * np.cos(THETA) + np.sqrt(1 - rho * rho) * np.sin(THETA))
    ax.plot(x, y, '-', color=color, linewidth=2)

In [ ]:
# Adjustable parameters
year = 2017#2022
name = "IRMA"#"IAN"
basin = 'AL'
ftimes = [12, 24, 36, 48, 60, 72, 96, 120]
rng_seeds = [1, 2, 3]

In [ ]:
# Gather the data for the specified hurricane.
data = pd.DataFrame(columns=['MMDDHH', 'sigma_u', 'sigma_v', 'rho', 'ftime', 'rng_seed'])

for ftime in ftimes:
    for rng_seed in rng_seeds:
        pathname = f"{PREDICTIONS_PATH}/centered_bivariate_normal_*_AL{ftime}_{year}_centered_bivariate_normal_rng_seed_{rng_seed}_testing_predictions.csv"
        file_list = glob.glob(pathname=pathname)

        # See if the file exists, if it does, load it.
        try:
            PREDICTIONS_FILE = file_list[0]
            predictions = pd.read_csv(PREDICTIONS_FILE)
        except:
            continue

        parameters = predictions.loc[predictions['NAME']==name, ['MMDDHH', 'sigma_u', 'sigma_v', 'rho']].copy()
        parameters["ftime"] = ftime
        parameters["rng_seed"] = rng_seed
        parameters.sort_values(by='MMDDHH', inplace=True, ignore_index=True)

        data = pd.concat([data, parameters], ignore_index=True)

times = data.time.unique()

In [ ]:
nrows = len(times)
ncols = 8

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(FIGURE_WIDTH, FIGURE_WIDTH * nrows/ncols),
    facecolor="white",
)

for i, time in enumerate(times):
    for j, ftime in enumerate(ftimes):
        axes[i, j].set_aspect('equal')
        axes[i, j].set_title(f"{time}:{ftime} {name} {year}")

        for rng_seed in rng_seeds:
            subset = data.loc[
                (data["MMDDHH"]==time) & (data["ftime"]==ftime) & (data["rng_seed"]==rng_seed),
                ['sigma_u', 'sigma_v', 'rho']
            ]

            for k, row in subset.iterrows():
                plot_ellipse(
                    axes[i, j],
                    row['sigma_u'],
                    row['sigma_v'],
                    row['rho'],
                    ELLIPSE_COLOR[rng_seed]
                )
        axes[i,j].set_ylim(-400,400)
        axes[i,j].set_xlim(-400,400)
        axes[i,j].set_xticks(np.arange(-400,400+200,200))
        axes[i,j].set_yticks(np.arange(-400,400+200,200))

# plt.tight_layout()
exp_name = f"centered_bivariate_normal_{name}_AL_{year}_centered_bivariate_normal"
plt.savefig(
    FIGURE_PATH + 'compare_rng_seeds_' + exp_name + '.png',
    dpi=dpiFig,
    bbox_inches='tight',
)
plt.show()